# Lab 4: LLMs and Prompt Engineering for Decision Support

**Duration:** 2 weeks [30 Jul - 13 Aug, 2026]
**Due Date:** 13th August, 2026
**Format:** Jupyter Notebook / Google Colab + external APIs + GitHub version control
**Grading:** This is a graded lab.

**Student Name:** [Enter Name]
**Student ID:** [Enter ID]

---

### Objective

In the previous labs you *trained* models. In this lab you will *use* a model that someone
else spent millions of dollars training — a **Large Language Model (LLM)** — and learn that
getting good results out of one is an engineering discipline of its own: **prompt
engineering**.

You will build a **decision support system for a microfinance loan officer**. Given a pile of
free-text loan application letters, your system will:

1. **Summarize** each application into a short, factual brief,
2. **Extract** specific structured data points (JSON) that a downstream system could store,
3. Produce a **decision-support recommendation** — while keeping the human firmly in the loop.

Just as importantly, you will **evaluate** the LLM's output for quality, reliability, and
appropriateness: Does it hallucinate? Is it consistent across runs? Should it be trusted to
make the final call?

---

### Choosing an API provider

You need an LLM API with a **free tier**. Recommended options (pick ONE):

| Provider | Free tier | Notes |
|---|---|---|
| **Groq** (recommended) | Yes, generous | OpenAI-compatible API, very fast, open models (Llama) |
| **Google Gemini** | Yes | `google-generativeai` package |
| **Hugging Face Inference API** | Yes, limited | Many open models |
| OpenAI / Anthropic | Paid | Fine if you already have credits |

The notebook's example code uses the **OpenAI-compatible chat format** (works with Groq and
OpenAI directly; Gemini users adapt the call in one place). Everything else in the lab is
provider-agnostic.

---
### Part 0: Repository and API-key setup

1. Create a **public** repository named `lab-4-llm-decision-support` and save this notebook
   inside it.
2. Sign up with your chosen provider and create an **API key**.
3. **NEVER hard-code or commit your API key.** This is a graded requirement.
   - Locally: put it in a `.env` file and add `.env` to `.gitignore`.
   - Colab: use the Secrets panel (key icon) and read it with `google.colab.userdata`.
4. Add a `requirements.txt`: `openai python-dotenv pandas matplotlib`.
5. Commit and push after **each Part** — we will check for incremental commits.

> **A leaked key in your commit history = resubmission + penalty.** Keys can be scraped from
> public repos within minutes.

In [6]:
import os

In [7]:
print(os.listdir())

['.config', 'sample_data']


In [8]:
# API-key setup — DO NOT hard-code your key in this cell.

import os

# --- Local (with a .env file) ---
# from dotenv import load_dotenv
# load_dotenv()
# API_KEY = os.environ.get("GROQ_API_KEY")

# --- Google Colab (Secrets panel) ---
from google.colab import userdata
API_KEY = userdata.get("GROQ_API_KEY")

if not API_KEY:
    raise ValueError(
        "GROQ_API_KEY not found. Create a .env file locally with "
        "GROQ_API_KEY=your_key_here (and add .env to .gitignore), "
        "or set it in the Colab Secrets panel."
    )

# OpenAI-compatible client (works for Groq and OpenAI; Gemini users see their docs):
from openai import OpenAI

client = OpenAI(
    api_key=API_KEY,
    base_url="https://api.groq.com/openai/v1",   # remove this line if using OpenAI itself
)
MODEL = "llama-3.3-70b-versatile"                # or your provider's model name

print("Client ready.")

Client ready.


---
# Section 1 — Talking to an LLM Programmatically

Before building anything, understand the anatomy of an API call: **messages and roles**
(`system`, `user`, `assistant`), and the **generation parameters** (`temperature`,
`max_tokens`).

### Part 1.1 — Your first API call

In [9]:
def ask_llm(user_prompt, system_prompt="You are a helpful assistant.",
            temperature=0.7, max_tokens=500):
    """Reusable helper: sends one user prompt (with an optional system prompt) to the LLM
    and returns just the text of the reply."""
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt},
        ],
        temperature=temperature,
        max_tokens=max_tokens,
    )
    return response.choices[0].message.content

# Raw call (not through the helper) so we can see the full response object
raw_response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "In one sentence, what does a microfinance loan officer do?"},
    ],
    temperature=0.7,
    max_tokens=200,
)
print("Answer:", raw_response.choices[0].message.content)
print("\nToken usage:", raw_response.usage)

# Confirm the helper gives the same kind of result
print("\nHelper output:", ask_llm("In one sentence, what does a microfinance loan officer do?"))

Answer: A microfinance loan officer is responsible for evaluating and approving small loans to low-income individuals or entrepreneurs, often in developing countries, and providing financial guidance and support to help them manage their debt and achieve financial stability.

Token usage: CompletionUsage(completion_tokens=43, prompt_tokens=54, total_tokens=97, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.008786083, prompt_time=0.002287202, completion_time=0.136554109, total_time=0.138841311)

Helper output: A microfinance loan officer works with low-income individuals or small businesses to provide access to financial services, such as small loans, savings accounts, and other financial products, and is responsible for evaluating creditworthiness, disbursing loans, and collecting repayments.


**Student Reasoning — Anatomy of a call**
*1. What is the difference between the `system` and `user` roles? Give an example of
something that belongs in each.*
*2. What is a token, roughly? Why do API providers bill per token rather than per request?*

> **Answer:** The system role sets the model's behavior, persona, and constraints for the whole conversation — it's instructions about how to answer, not a question itself. The user role is the actual input/question you want answered.

Example: in my ask_llm calls, the system prompt was "You are a helpful assistant." — this frames who the model should act as. The user prompt was "In one sentence, what does a microfinance loan officer do?" — this is the actual task. Later in the lab, the system prompt gets more specific (e.g. "You are an assistant to a microfinance loan officer... do not invent details"), while the user prompt stays focused on just the letter to summarize. Separating them this way means I can change the rules (system) without rewriting the question (user) every time.

2. What is a token, roughly? Why bill per token?

A token is roughly a chunk of text — often a word, part of a longer word, or a punctuation mark — that the model processes as one unit. It's not the same as a character or a word; "microfinance" might be split into 2-3 tokens, while "the" is usually one.

Providers bill per token rather than per request because the actual computational cost of a call scales with how much text goes in and comes out, not with how many calls are made. My own call used prompt_tokens=54 and completion_tokens=43 — a short question with a longer answer costs more than a short question with a short answer, even though both are "one request." Billing per request would charge the same for a one-word answer and a 500-word essay, which doesn't reflect the actual compute used.

### Part 1.2 — Temperature: the randomness dial

In [10]:
import textwrap

question = "Suggest a name for a savings product for market traders in Accra."

results = {0.0: [], 1.2: []}
for temp in results:
    for i in range(5):
        answer = ask_llm(question, temperature=temp, max_tokens=60)
        results[temp].append(answer)

for temp, answers in results.items():
    print(f"\n=== Temperature = {temp} ===")
    for i, ans in enumerate(answers, 1):
        print(f"{i}. {textwrap.shorten(ans, 200)}")


=== Temperature = 0.0 ===
1. Here are a few suggestions for a savings product for market traders in Accra: 1. **Makola Save**: "Makola" is a well-known market in Accra, so this name could resonate with market traders. 2. [...]
2. Here are a few suggestions for a savings product for market traders in Accra: 1. **Makola Save**: "Makola" is a well-known market in Accra, so this name could resonate with market traders. 2. [...]
3. Here are a few suggestions for a savings product for market traders in Accra: 1. **Makola Save**: "Makola" is a well-known market in Accra, so this name could resonate with market traders. 2. [...]
4. Here are a few suggestions for a savings product for market traders in Accra: 1. **Makola Save**: "Makola" is a well-known market in Accra, so this name could resonate with market traders. 2. [...]
5. Here are a few suggestions for a savings product for market traders in Accra: 1. **Makola Save**: "Makola" is a well-known market in Accra, so this name could resonat

**Student Reasoning — Temperature**
*What did you observe at each temperature? For the loan decision-support system you are about
to build, which temperature regime is appropriate, and why?*

> **Answer:** At temperature 0.0, all 5 answers came out exactly the same — same wording, same first suggestion, same everything. At temperature 1.2, the answers were different from each other. They still all kept coming back to "Makola" (the market name), but the wording changed each time and one answer even came up with a totally different name using a Twi word instead of English.

For the loan decision-support system you are about to build, which temperature regime is appropriate, and why?

I think temperature 0.0 makes the most sense for this system. Since we're dealing with things like loan amounts, repayment terms, and risk assessments for real applicants, I don't want the answers to change randomly every time I run the same letter through the system. If temperature were high, the same application could get summarized or evaluated differently each time, which isn't fair to applicants and would make it hard to trust the tool. A high temperature might be fine for something more creative, like brainstorming product names, but for anything in the actual loan process I think consistency matters more than variety.

---
# Section 2 — The Dataset: Loan Application Letters

Run the next cell to load **six loan application letters** submitted to a (fictional)
microfinance institution in Ghana, plus **gold-standard extraction labels** for three of them
(you will use these for evaluation in Section 4).

Read at least two letters fully before moving on — you cannot engineer prompts for text you
have not read.

In [11]:
LETTERS = {
"L001": """Dear Sir/Madam,
My name is Akosua Mensah and I have been selling provisions at Makola Market for 12 years.
I am applying for a loan of GHS 8,000 to buy a deep freezer and expand into frozen foods.
My current stall makes about GHS 900 profit each month. I have saved GHS 2,500 with your
susu scheme over the past two years and I have never missed a contribution. I can repay
GHS 450 monthly over 20 months. My sister, a teacher, will stand as my guarantor.
Thank you for considering my application.""",

"L002": """Hello,
I am Kwame Boateng, a commercial driver in Kumasi. I need GHS 25,000 urgently to repair my
trotro engine and settle some personal debts. Business has been slow but it will surely
pick up after the festive season. I can pay back whenever the money comes. I do not have
collateral at the moment but God willing everything will be fine. Please help me quickly.""",

"L003": """Dear Loan Committee,
I am Efua Darko, owner of Darko Fashions, a registered dressmaking business in Takoradi
(registration no. BN-2019-4482). I employ three apprentices. I request GHS 15,000 to
purchase two industrial sewing machines and fabric stock ahead of the Christmas season.
Last year my December revenue alone was GHS 22,000; monthly profit averages GHS 2,800.
I hold a fixed deposit of GHS 5,000 with GCB which I can pledge. Proposed repayment:
GHS 1,100 monthly for 15 months. Attached are my sales records for the past 18 months.""",

"L004": """Good day,
My name is Yaw Owusu. I want a loan for my poultry farm at Nsawam. The amount is GHS 12,000
for feed and 500 new layers. I started the farm last year. Sometimes I make good money,
around GHS 1,500 in a good month, but bird flu affected us in March and I lost many birds.
I am rebuilding now. I can repay in 18 months. My uncle has agreed to guarantee the loan
with his taxi.""",

"L005": """Dear Manager,
I am writing on behalf of the Adenta Women's Weaving Cooperative (14 members). We seek
GHS 30,000 to buy a bulk order of yarn directly from the factory, cutting out middlemen and
raising our margins from 15% to about 35%. The cooperative has operated for 6 years and
holds GHS 9,000 in our group account. We propose repayment of GHS 2,000 monthly over
16 months, backed by our group savings and joint liability agreement.""",

"L006": """Hi,
This is Kofi. I saw your advert. I want GHS 50,000 to start a car washing business, a
provision shop, and also import phones from Dubai. I am 22 and full of energy. I have not
started any of these yet but my friends say I am very business minded. I will pay back in
one year when the businesses are booming. No collateral but I am trustworthy.""",
}

# Gold-standard labels for three letters (for Section 4 evaluation):
GOLD = {
  "L001": {"applicant_name": "Akosua Mensah", "amount_ghs": 8000,  "purpose": "buy deep freezer / expand into frozen foods",
           "monthly_profit_ghs": 900,  "has_collateral_or_guarantor": True,  "repayment_months": 20},
  "L003": {"applicant_name": "Efua Darko",    "amount_ghs": 15000, "purpose": "industrial sewing machines and fabric stock",
           "monthly_profit_ghs": 2800, "has_collateral_or_guarantor": True,  "repayment_months": 15},
  "L006": {"applicant_name": "Kofi",          "amount_ghs": 50000, "purpose": "car wash, provision shop, phone imports",
           "monthly_profit_ghs": None, "has_collateral_or_guarantor": False, "repayment_months": 12},
}

print(f"{len(LETTERS)} letters loaded.")

6 letters loaded.


---
# Section 3 — Prompt Engineering for the Decision Support System

You will now build the three components of the system, iterating on your prompts as you go.
**Keep every major prompt version** — Section 3.4 asks you to commit your prompt templates
and document how they evolved.

### Part 3.1 — Component 1: Summarization
Turn a rambling letter into a 3-4 sentence factual brief a busy loan officer can scan.

In [12]:
# --- V1: naive prompt ---
SUMMARY_PROMPT_V1 = "Summarize this:"

for letter_id in ["L002", "L006"]:
    letter = LETTERS[letter_id]
    v1_out = ask_llm(f"{SUMMARY_PROMPT_V1}\n\n{letter}", temperature=0)
    print(f"--- {letter_id} (V1) ---\n{v1_out}\n")

# --- V2: role + constraints in the system prompt, clean user template ---
SUMMARY_SYSTEM_V2 = (
    "You are an assistant to a microfinance loan officer. Summarize loan application "
    "letters factually and neutrally in 3-4 sentences. Do not invent, assume, or embellish "
    "any detail that is not explicitly stated in the letter. Do not give an opinion on "
    "whether the loan should be approved."
)

def SUMMARY_PROMPT_V2(letter_text):
    return f"Summarize this loan application:\n\n{letter_text}"

for letter_id in ["L002", "L006"]:
    letter = LETTERS[letter_id]
    v2_out = ask_llm(SUMMARY_PROMPT_V2(letter), system_prompt=SUMMARY_SYSTEM_V2, temperature=0)
    print(f"--- {letter_id} (V2) ---\n{v2_out}\n")

--- L002 (V1) ---
Kwame Boateng, a commercial driver in Kumasi, is urgently seeking GHS 25,000 to repair his vehicle's engine and pay off personal debts. He's experiencing a slow business period but expects it to improve after the festive season and is willing to repay the loan when his finances improve.

--- L006 (V1) ---
Kofi, a 22-year-old, is seeking a loan of GHS 50,000 to start three businesses: a car washing service, a provision shop, and a phone import business from Dubai. He has no prior experience, but claims to be "business-minded" based on his friends' opinions. He promises to repay the loan within a year, once his businesses are successful, and offers his trustworthiness as assurance, since he has no collateral to provide.

--- L002 (V2) ---
Kwame Boateng, a commercial driver in Kumasi, has applied for a loan of GHS 25,000. He intends to use the funds to repair his trotro engine and settle personal debts. Mr. Boateng mentions that his business has been slow, but he expects

**Student Reasoning — Summarization prompts**
*1. What concrete problems did V1's output have that V2 fixed? Quote examples.*
*2. Why is "no invented details" an essential instruction in this application? What is this
failure mode called in the LLM literature?*

> **Answer:** The main problem I noticed is that V1 added its own interpretation instead of sticking to just the facts. For example, in the L002 summary, V1 says Kwame is "urgently seeking" the loan — but the letter doesn't say he's "urgent" about it, that's just the model's own spin on the situation. V2's version of the same letter doesn't do this; it just says he "has applied for a loan," which is more neutral.

Another problem is that V1 left out important details inconsistently. For L002, V1 never mentions whether Kwame has collateral at all. V2 explicitly says "he does not currently have collateral to offer" for the same letter. Since collateral is something a loan officer would actually need to know, V1 dropping it silently is a real problem, not just a style issue.

Basically, V1 had no instructions telling it what to focus on or what to avoid, so it just summarized however it felt like, which meant sometimes adding opinion-like language and sometimes skipping over facts that mattered.

2. Why is "no invented details" an essential instruction in this application? What is this failure mode called in the LLM literature?

This instruction matters because this system is meant to support real decisions about real people's loan applications. If the model adds details that aren't actually in the letter — or exaggerates something like how "urgent" a request is — the loan officer could end up making a decision based on information that was never actually stated by the applicant. That's not just inaccurate, it could be unfair to the applicant if the made-up detail makes them look better or worse than they actually are.

This failure mode is called "hallucination" in the LLM literature — when a model generates text that sounds plausible and confident but isn't actually grounded in the source material it was given.

### Part 3.2 — Component 2: Structured extraction (JSON)
Downstream software cannot read prose. Extract the fields in `GOLD` as strict JSON.

In [13]:
import json
import pandas as pd

# One worked example NOT taken from LETTERS, used as a few-shot demonstration.
FEWSHOT_LETTER = """Dear Sir,
My name is John Mensah. I run a small carpentry workshop in Tema and have been in business
for 5 years. I am requesting a loan of GHS 10,000 to purchase a new wood-cutting machine.
My monthly profit is about GHS 1,200. I own my workshop building outright, which I can offer
as collateral. I propose to repay GHS 700 monthly over 15 months."""

FEWSHOT_JSON = {
    "applicant_name": "John Mensah",
    "amount_ghs": 10000,
    "purpose": "purchase a new wood-cutting machine",
    "monthly_profit_ghs": 1200,
    "has_collateral_or_guarantor": True,
    "repayment_months": 15,
}

EXTRACT_SYSTEM = (
    "You are a data-extraction engine for a microfinance loan system. You return ONLY a "
    "single valid JSON object and nothing else -- no markdown fences, no commentary."
)

EXTRACT_PROMPT_TEMPLATE = """Extract the following fields from the loan application letter below,
and return ONLY a JSON object with EXACTLY these keys:

- applicant_name (string)
- amount_ghs (number)
- purpose (string)
- monthly_profit_ghs (number or null)
- has_collateral_or_guarantor (boolean)
- repayment_months (number or null)

If a field is not explicitly stated in the letter, use null. Do not guess or infer a value
that is not stated.

Example letter:
{fewshot_letter}

Example output:
{fewshot_json}

Now extract from this letter:
{letter_text}
"""

def extract_fields(letter_text):
    """Calls the LLM, strips ```json fences if present, parses the JSON, and returns a dict.
    Returns None (with a printed warning) if parsing fails."""
    prompt = EXTRACT_PROMPT_TEMPLATE.format(
        fewshot_letter=FEWSHOT_LETTER,
        fewshot_json=json.dumps(FEWSHOT_JSON),
        letter_text=letter_text,
    )
    raw = ask_llm(prompt, system_prompt=EXTRACT_SYSTEM, temperature=0, max_tokens=300)

    cleaned = raw.strip()
    if cleaned.startswith("```"):
        cleaned = cleaned.strip("`")
        if cleaned.lower().startswith("json"):
            cleaned = cleaned[4:].strip()

    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        print(f"WARNING: could not parse JSON for this letter. Raw output was:\n{raw}")
        return None

rows = []
for letter_id, letter_text in LETTERS.items():
    fields = extract_fields(letter_text)
    row = {"letter_id": letter_id}
    row.update(fields if fields else {})
    rows.append(row)

extracted_df = pd.DataFrame(rows).set_index("letter_id")
extracted_df

,applicant_name,amount_ghs,purpose,monthly_profit_ghs,has_collateral_or_guarantor,repayment_months
letter_id,,,,,,
L001,Akosua Mensah,8000,buy a deep freezer and expand into frozen foods,900.0,True,20.0
L002,Kwame Boateng,25000,repair my trotro engine and settle some person...,NaN,False,NaN
L003,Efua Darko,15000,purchase two industrial sewing machines and fa...,2800.0,True,15.0
L004,Yaw Owusu,12000,for feed and 500 new layers,1500.0,True,18.0
L005,Adenta Women's Weaving Cooperative,30000,buy a bulk order of yarn directly from the fac...,NaN,True,16.0
L006,Kofi,50000,"start a car washing business, a provision shop...",NaN,False,12.0


**Student Reasoning — Structured extraction**
*1. Why must the few-shot example NOT come from the six letters you are processing?*
*2. Why "use null, do not guess" — what did the model do without that instruction?*
*3. Why is temperature=0 the right choice for extraction but arguably not for creative tasks?*

> **Answer:** If I used one of the six real letters as my few-shot example, I'd be showing the model the exact answer to one of the items I'm later going to evaluate for accuracy. That would make my Part 4.1 accuracy check meaningless for that letter, since the model wouldn't really be "extracting" — it would just be copying an answer I already gave it in the prompt. Using a made-up letter (John Mensah) keeps the six real letters as a genuinely unseen test set.

2. Why "use null, do not guess" — what did the model do without that instruction?

Without this instruction, the model would likely try to fill in every field even when the letter doesn't actually state a value, because it's trained to give complete, helpful-sounding answers. For example, if a letter never mentions monthly profit, a model without this rule might guess a plausible-sounding number instead of admitting the information isn't there. With the instruction in place, letters like L002, L005, and L006 correctly came back with null/NaN for monthly_profit_ghs instead of a made-up figure. This matters a lot for a loan system, since a fabricated number could look just as convincing as a real one.

3. Why is temperature=0 the right choice for extraction but arguably not for creative tasks?

Extraction is supposed to have one correct answer — the letter either says GHS 8,000 or it doesn't, so there's no reason to want variety in the output. Temperature=0 makes the model pick the most likely token every time, which keeps extraction consistent and repeatable, and repeatability matters if the same letter could be processed more than once. For a creative task, like brainstorming a product name, there isn't one "correct" answer, so some randomness from a higher temperature is actually useful — it gives more options to choose from instead of the same single suggestion every time.

### Part 3.3 — Component 3: The decision-support brief
Combine everything: for each letter, produce a recommendation brief for the loan officer —
strengths, risks, missing information, and a suggested next step. The system must
**support** the decision, not **make** it.

In [14]:
BRIEF_SYSTEM = (
    "You are a decision-support assistant for a human microfinance loan officer. You "
    "NEVER approve or reject a loan yourself -- the human officer always makes the final "
    "decision. Base every point strictly on the letter and extracted data provided; do not "
    "invent facts that are not present in them."
)

BRIEF_PROMPT_TEMPLATE = """Here is a loan application letter and the structured data extracted from it.

Letter:
{letter_text}

Extracted data:
{extracted_json}

Write a decision-support brief for the loan officer with these four sections:
1. Strengths (bullet points, grounded in the letter)
2. Risks / red flags (bullet points)
3. Missing information the officer should request
4. Suggested next step (e.g. "invite for interview", "request documents", "flag for senior
   review") -- do NOT write "approve" or "reject". The final decision belongs to the human
   officer.
"""

def make_brief(letter_id):
    letter_text = LETTERS[letter_id]
    extracted = extracted_df.loc[letter_id].to_dict()
    prompt = BRIEF_PROMPT_TEMPLATE.format(
        letter_text=letter_text,
        extracted_json=json.dumps(extracted, default=str),
    )
    return ask_llm(prompt, system_prompt=BRIEF_SYSTEM, temperature=0, max_tokens=500)

briefs = {letter_id: make_brief(letter_id) for letter_id in LETTERS}

for letter_id in ["L001", "L002", "L006"]:
    print(f"===== Brief for {letter_id} =====\n{briefs[letter_id]}\n")


===== Brief for L001 =====
**Decision-Support Brief**

### 1. Strengths
* The applicant, Akosua Mensah, has 12 years of experience selling provisions at Makola Market, indicating a stable business history.
* She has a proven track record of saving with the susu scheme, having saved GHS 2,500 over two years without missing a contribution, demonstrating financial discipline.
* The applicant has a guarantor, her sister, who is a teacher, potentially providing an additional layer of financial security.
* Akosua Mensah has a clear plan for using the loan, which is to buy a deep freezer and expand into frozen foods, suggesting a thought-out business strategy.

### 2. Risks / Red Flags
* The loan amount of GHS 8,000 is significant compared to the applicant's monthly profit of GHS 900, which might pose a repayment risk if the business does not expand as planned.
* There is no detailed information on how the expansion into frozen foods will increase profits, making it difficult to assess the vi

In [15]:
print(briefs["L003"])

**Decision-Support Brief**

### 1. Strengths
* The applicant has a registered business with a proven track record, as evidenced by the provided registration number and sales records for the past 18 months.
* The business has a significant revenue potential, with December revenue alone being GHS 22,000.
* The applicant has a stable monthly profit average of GHS 2,800.
* The applicant is willing to pledge a fixed deposit of GHS 5,000 as collateral.

### 2. Risks / Red Flags
* The loan amount of GHS 15,000 is relatively large compared to the applicant's monthly profit, which may pose a repayment risk.
* The repayment plan of GHS 1,100 monthly for 15 months may be challenging for the business to sustain, especially if profits fluctuate.
* There is no information provided about the applicant's credit history or previous loan repayments.

### 3. Missing Information
* Detailed breakdown of the costs for the industrial sewing machines and fabric stock.
* Information about the applicant's credi

**Student Reasoning — Decision support**
*1. Compare the briefs for L003 (strong application) and L006 (weak application). Did the
system identify the right strengths and red flags in each?*
*2. Why did we forbid the model from outputting "approve"/"reject"? Give one practical and
one ethical reason.*

> **Answer:** Yes, I think the system picked up the right signals in both cases. For L003, the strengths it listed were concrete and verifiable: a registered business, 18 months of sales records, a stated monthly profit of GHS 2,800, and GHS 5,000 offered as collateral. These are all things that actually appear in the extracted data (has_collateral_or_guarantor: True, monthly_profit_ghs: 2800), so the brief is grounded in real numbers, not just a general impression that the applicant "seems good."

For L006, the strengths section is noticeably weaker, and I think that's actually accurate rather than a mistake — Kofi doesn't have a registered business, sales history, or collateral, so the model could only point to things like "young and energetic" and "confidence endorsed by friends," which aren't real financial strengths. The red flags reflect this too: L006's risks include no experience, no collateral, and a repayment plan that depends on the business "booming," while L003's risks are more moderate, like the loan amount being large relative to profit. So the system didn't just generate generic praise for every applicant — it scaled the strength of the strengths section to how much real evidence was actually in the letter, which is what I'd want a decision-support tool to do.

One thing I noticed: for L006, listing "confidence in their business acumen, as endorsed by their friends" as a strength is a little questionable, since that's really just the applicant's own unverified claim, not independent evidence. If I were revising this, I might want the system to be more skeptical about restating an applicant's self-description as a strength.

2. Why did we forbid the model from outputting "approve"/"reject"? Give one practical and one ethical reason.

Practical reason: the model only has access to the letter and a few extracted fields — it doesn't have the lender's actual credit policy, risk thresholds, portfolio constraints, or ability to verify any of the claims in the letter (like whether the sales records are real). Letting it output "approve" or "reject" would mean making a final financial decision based on incomplete information, when a human officer has access to more context and can actually verify documents.

Ethical reason: loan decisions affect real people's access to money and opportunity, and an LLM can be wrong or biased in ways that aren't obvious from the outside. If the system directly said "approve" or "reject," an officer might just defer to it without applying their own judgment, especially over time — that shifts real accountability for a high-stakes decision onto a tool that can hallucinate or misjudge risk. Keeping the model at the level of "here's what to look at" instead of "here's the answer" keeps a human responsible for the actual decision.

### Part 3.4 — Commit your prompt templates
Prompts ARE code. Save your final `SUMMARY_PROMPT`, `EXTRACT_PROMPT`, and `BRIEF_PROMPT` into
a separate file `prompts.py` (or `prompts.md`) in your repository and commit it with a
message describing how the prompts evolved. Paste your commit hash below.

> **Commit hash:** [paste here]

---
# Section 4 — Evaluation: Quality, Reliability, Appropriateness

An impressive demo is not a trustworthy system. Now measure it.

### Part 4.1 — Extraction accuracy against gold labels

In [ ]:
def fields_match(field, predicted, gold_value):
    if predicted is None and gold_value is None:
        return True
    if predicted is None or gold_value is None:
        return False
    if field == "applicant_name":
        return isinstance(predicted, str) and predicted.strip().lower() == gold_value.strip().lower()
    if field == "purpose":
        # purpose is free text -- count it a match if the extracted text shares at least
        # two meaningful words with the gold description.
        if not isinstance(predicted, str):
            return False
        gold_words = set(gold_value.lower().split())
        pred_words = set(predicted.lower().split())
        return len(gold_words & pred_words) >= 2
    return predicted == gold_value

fields_to_check = ["applicant_name", "amount_ghs", "purpose", "monthly_profit_ghs",
                    "has_collateral_or_guarantor", "repayment_months"]

comparison_rows = []
for field in fields_to_check:
    row = {"field": field}
    matches = 0
    for letter_id, gold in GOLD.items():
        predicted = extracted_df.loc[letter_id, field] if field in extracted_df.columns else None
        is_match = fields_match(field, predicted, gold[field])
        row[letter_id] = "correct" if is_match else "wrong"
        matches += int(is_match)
    row["accuracy"] = f"{matches}/{len(GOLD)}"
    comparison_rows.append(row)

accuracy_df = pd.DataFrame(comparison_rows).set_index("field")
accuracy_df


### Part 4.2 — Reliability: is the system consistent?

In [ ]:
def run_reliability_test(letter_id, temperature, n_runs=5):
    outcomes = []
    valid_json_count = 0
    for _ in range(n_runs):
        prompt = EXTRACT_PROMPT_TEMPLATE.format(
            fewshot_letter=FEWSHOT_LETTER,
            fewshot_json=json.dumps(FEWSHOT_JSON),
            letter_text=LETTERS[letter_id],
        )
        raw = ask_llm(prompt, system_prompt=EXTRACT_SYSTEM, temperature=temperature, max_tokens=300)
        cleaned = raw.strip().strip("`")
        if cleaned.lower().startswith("json"):
            cleaned = cleaned[4:].strip()
        try:
            parsed = json.loads(cleaned)
            valid_json_count += 1
            outcomes.append(json.dumps(parsed, sort_keys=True))
        except json.JSONDecodeError:
            outcomes.append(None)

    unique_outcomes = set(o for o in outcomes if o is not None)
    return {
        "temperature": temperature,
        "valid_json": f"{valid_json_count}/{n_runs}",
        "identical_across_all_runs": valid_json_count == n_runs and len(unique_outcomes) == 1,
        "n_unique_outputs": len(unique_outcomes),
    }

reliability_results = [
    run_reliability_test("L004", temperature=0.0),
    run_reliability_test("L004", temperature=1.0),
]
pd.DataFrame(reliability_results)


### Part 4.3 — Hallucination probing

In [ ]:
# --- Test 1: ask about a detail that is NOT in the letter ---
test1_prompt = "Based only on this letter, what is the applicant's credit score?\n\n" + LETTERS["L001"]
test1_output = ask_llm(
    test1_prompt,
    system_prompt=(
        "Answer using ONLY information contained in the letter. If the requested "
        "information is not present, say clearly and explicitly that it is not stated."
    ),
    temperature=0,
)
print("TEST 1 OUTPUT:\n", test1_output)
test1_pass = any(p in test1_output.lower() for p in ["not stated", "not mention", "no mention", "does not", "doesn't", "not provided", "not given"])
print("Result:", "PASS" if test1_pass else "FAIL")

# --- Test 2: feed the extractor something irrelevant ---
irrelevant_text = """Weather report for Accra, 12 August 2026: Skies partly cloudy with a
high of 31C and a low of 24C. Light showers expected in the afternoon. Humidity around 78%."""

test2_output = extract_fields(irrelevant_text)
print("\nTEST 2 OUTPUT:\n", test2_output)
test2_pass = (
    test2_output is None
    or all(v in (None, False) for v in test2_output.values())
)
print("Result:", "PASS" if test2_pass else "FAIL")


**Student Reasoning — Evaluation results**
*1. Report your extraction accuracy. Which field was hardest for the model and why?*
*2. What did the reliability experiment show about temperature and production systems?*
*3. Did your system hallucinate under probing? If yes, how could the prompt (or the system
design around it) reduce the risk?*

> **Answer:** [Double-click to edit]

### Part 4.4 — Appropriateness: should this system exist?
No code in this part — just judgment, which is the scarcest skill in AI for business.

**Student Reasoning — Appropriateness**
*1. Letters L002 and L006 would likely be declined. If the bank fully automated decisions
with your system, who could be unfairly harmed, and how? Consider applicants who write
poorly in English but run solid businesses.*
*2. Loan letters contain personal data. What are the implications of sending them to a
third-party API in another country? What would you check before deploying this at a real
Ghanaian microfinance institution?*
*3. Name TWO concrete safeguards you would build around this system in production (think:
human review points, logging, appeal processes, monitoring).*

> **Answer:** [Double-click to edit]

---
# Section 5 — Reflection

*Answer in a few sentences each:*

1. **Prompting as engineering:** How is iterating on a prompt similar to and different from
   iterating on the model hyperparameters you tuned in Lab 3?
2. **Trust:** After your Section 4 evaluation, would you trust this system to run unattended?
   What single evaluation result most influenced your answer?
3. **Cost and scale:** Estimate (from your `response.usage` numbers) the tokens needed to
   process 1,000 applications per month. What does that imply for provider choice?
4. **Looking back at the course:** You have now used classical ML (Lab 2), trained neural
   networks (Lab 3), and used a foundation model via API (Lab 4). For a task like this one,
   why does calling an API beat training your own model — and when would it not?

> **Answer:** [Double-click to edit]

---
### Submission checklist

- [ ] All cells run top-to-bottom with no errors (`Kernel -> Restart & Run All`).
- [ ] **No API key anywhere in the notebook or the commit history.**
- [ ] Every **Student Reasoning** box is filled in with full sentences.
- [ ] `prompts.py` / `prompts.md` committed with your final prompt templates.
- [ ] Evaluation tables and adversarial test outputs visible in the saved notebook.
- [ ] Notebook pushed to `lab-4-llm-decision-support` with incremental commits.
- [ ] Repository link submitted to the course portal.
- [ ] AI Declaration form in Repository.